# Fase 1: Ingestão de Dados e Análise Exploratória (EDA)
**Objetivo:** Realizar a leitura do dataset de 1 milhão de funcionários, tratar inconsistências e analisar os padrões das variáveis em relação à taxa de Attrition.

In [0]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels missingno -q
!pip install lightgbm

In [0]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

sns.set_theme(style="whitegrid", context="notebook")

In [0]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def clip_int(x, low, high):
    return np.clip(np.round(x), low, high).astype(int)


def generate_ibm_hr_synthetic(n=1470, seed=42):
    """
    Gera uma base sintética inspirada no IBM HR Analytics Employee Attrition & Performance.

    A variável Attrition é gerada por uma função logística com efeitos interpretáveis:
    - maior attrition: OverTime, baixa satisfação, maior distância, menor idade,
      menor renda, solteiro, viagem frequente, pouco tempo na empresa.
    - menor attrition: maior senioridade, maior renda, mais estabilidade,
      stock option level, boa satisfação e bom equilíbrio vida-trabalho.
    """
    rng = np.random.default_rng(seed)

    employee_number = np.arange(1, n + 1)

    age = clip_int(rng.normal(37, 9, n), 18, 60)
    gender = rng.choice(["Male", "Female"], size=n, p=[0.60, 0.40])

    marital_status = []
    for a in age:
        if a < 28:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.58, 0.37, 0.05]))
        elif a < 42:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.23, 0.65, 0.12]))
        else:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.11, 0.69, 0.20]))
    marital_status = np.array(marital_status)

    department = rng.choice(
        ["Research & Development", "Sales", "Human Resources"],
        size=n,
        p=[0.65, 0.30, 0.05]
    )

    education = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.12, 0.20, 0.39, 0.24, 0.05])

    education_field = []
    for d in department:
        if d == "Research & Development":
            education_field.append(rng.choice(
                ["Life Sciences", "Medical", "Technical Degree", "Other"],
                p=[0.38, 0.35, 0.20, 0.07]
            ))
        elif d == "Sales":
            education_field.append(rng.choice(
                ["Marketing", "Life Sciences", "Other", "Technical Degree"],
                p=[0.52, 0.20, 0.20, 0.08]
            ))
        else:
            education_field.append(rng.choice(
                ["Human Resources", "Other", "Life Sciences"],
                p=[0.65, 0.25, 0.10]
            ))
    education_field = np.array(education_field)

    job_role = []
    for d in department:
        if d == "Research & Development":
            job_role.append(rng.choice(
                ["Research Scientist", "Laboratory Technician", "Manufacturing Director",
                 "Healthcare Representative", "Research Director", "Manager"],
                p=[0.34, 0.32, 0.12, 0.12, 0.04, 0.06]
            ))
        elif d == "Sales":
            job_role.append(rng.choice(
                ["Sales Executive", "Sales Representative", "Manager"],
                p=[0.70, 0.22, 0.08]
            ))
        else:
            job_role.append(rng.choice(
                ["Human Resources", "Manager"],
                p=[0.82, 0.18]
            ))
    job_role = np.array(job_role)

    total_working_years = clip_int((age - 18) * rng.uniform(0.45, 0.95, n) + rng.normal(0, 2.2, n), 0, 40)
    job_level_raw = 1 + (total_working_years / 9) + rng.normal(0, 0.7, n)
    job_level = clip_int(job_level_raw, 1, 5)

    years_at_company = np.array([
        rng.integers(0, max(1, twy + 1)) if twy > 0 else 0
        for twy in total_working_years
    ])
    years_at_company = np.minimum(years_at_company, total_working_years)

    years_in_current_role = np.array([
        rng.integers(0, yac + 1) if yac > 0 else 0
        for yac in years_at_company
    ])

    years_with_curr_manager = np.array([
        rng.integers(0, yac + 1) if yac > 0 else 0
        for yac in years_at_company
    ])

    years_since_last_promotion = np.array([
        rng.integers(0, min(yac + 1, 16)) if yac > 0 else 0
        for yac in years_at_company
    ])

    num_companies_worked = clip_int(rng.poisson(lam=np.maximum(1.2, total_working_years / 8)), 0, 9)
    distance_from_home = clip_int(rng.gamma(shape=2.0, scale=4.8, size=n), 1, 29)

    business_travel = rng.choice(
        ["Non-Travel", "Travel_Rarely", "Travel_Frequently"],
        size=n,
        p=[0.10, 0.70, 0.20]
    )

    environment_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.18, 0.19, 0.31, 0.32])
    job_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.17, 0.20, 0.32, 0.31])
    relationship_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.16, 0.21, 0.32, 0.31])
    job_involvement = rng.choice([1, 2, 3, 4], size=n, p=[0.08, 0.25, 0.50, 0.17])
    work_life_balance = rng.choice([1, 2, 3, 4], size=n, p=[0.08, 0.24, 0.45, 0.23])

    performance_rating = rng.choice([3, 4], size=n, p=[0.85, 0.15])
    percent_salary_hike = clip_int(rng.normal(13 + 3 * (performance_rating == 4), 3, n), 11, 25)

    stock_option_level = []
    for m in marital_status:
        if m == "Single":
            stock_option_level.append(rng.choice([0, 1, 2, 3], p=[0.58, 0.28, 0.10, 0.04]))
        else:
            stock_option_level.append(rng.choice([0, 1, 2, 3], p=[0.34, 0.41, 0.18, 0.07]))
    stock_option_level = np.array(stock_option_level)

    over_time = []
    for wlb, ji in zip(work_life_balance, job_involvement):
        p_ot = 0.18 + 0.08 * (wlb <= 2) + 0.05 * (ji >= 3)
        over_time.append(rng.choice(["No", "Yes"], p=[1 - p_ot, p_ot]))
    over_time = np.array(over_time)

    training_times_last_year = clip_int(rng.poisson(2.3, n), 0, 6)

    base_income = (
        1300
        + job_level * 2400
        + total_working_years * 105
        + (performance_rating == 4) * 600
        + rng.normal(0, 850, n)
    )

    role_bonus_map = {
        "Manager": 3000,
        "Research Director": 3500,
        "Manufacturing Director": 1300,
        "Healthcare Representative": 950,
        "Sales Executive": 850,
        "Research Scientist": 450,
        "Laboratory Technician": -250,
        "Sales Representative": -450,
        "Human Resources": 100
    }
    role_bonus = np.array([role_bonus_map[r] for r in job_role])

    monthly_income = clip_int(base_income + role_bonus, 1000, 22000)
    monthly_rate = clip_int(monthly_income * rng.uniform(1.2, 1.9, n) + rng.normal(0, 900, n), 2000, 27000)
    daily_rate = clip_int(rng.normal(800, 400, n), 100, 1500)
    hourly_rate = clip_int(rng.normal(66, 20, n), 30, 100)

    employee_count = np.ones(n, dtype=int)
    standard_hours = np.repeat(80, n)
    over_18 = np.repeat("Y", n)

    logit = (
        -1.85
        + 0.82 * (over_time == "Yes")
        + 0.52 * (business_travel == "Travel_Frequently")
        + 0.20 * (business_travel == "Travel_Rarely")
        + 0.42 * (marital_status == "Single")
        - 0.20 * (marital_status == "Married")
        + 0.024 * distance_from_home
        - 0.035 * (age - 35)
        - 0.000065 * (monthly_income - 6000)
        - 0.23 * (environment_satisfaction - 2.5)
        - 0.27 * (job_satisfaction - 2.5)
        - 0.18 * (work_life_balance - 2.5)
        - 0.19 * (job_involvement - 2.5)
        - 0.07 * years_at_company
        + 0.12 * num_companies_worked
        - 0.18 * stock_option_level
        + 0.28 * (job_role == "Sales Representative")
        + 0.18 * (job_role == "Laboratory Technician")
        - 0.10 * training_times_last_year
    )

    attrition_probability = sigmoid(logit)
    attrition = rng.binomial(1, attrition_probability)

    df = pd.DataFrame({
        "Age": age,
        "Attrition": np.where(attrition == 1, "Yes", "No"),
        "BusinessTravel": business_travel,
        "DailyRate": daily_rate,
        "Department": department,
        "DistanceFromHome": distance_from_home,
        "Education": education,
        "EducationField": education_field,
        "EmployeeCount": employee_count,
        "EmployeeNumber": employee_number,
        "EnvironmentSatisfaction": environment_satisfaction,
        "Gender": gender,
        "HourlyRate": hourly_rate,
        "JobInvolvement": job_involvement,
        "JobLevel": job_level,
        "JobRole": job_role,
        "JobSatisfaction": job_satisfaction,
        "MaritalStatus": marital_status,
        "MonthlyIncome": monthly_income,
        "MonthlyRate": monthly_rate,
        "NumCompaniesWorked": num_companies_worked,
        "Over18": over_18,
        "OverTime": over_time,
        "PercentSalaryHike": percent_salary_hike,
        "PerformanceRating": performance_rating,
        "RelationshipSatisfaction": relationship_satisfaction,
        "StandardHours": standard_hours,
        "StockOptionLevel": stock_option_level,
        "TotalWorkingYears": total_working_years,
        "TrainingTimesLastYear": training_times_last_year,
        "WorkLifeBalance": work_life_balance,
        "YearsAtCompany": years_at_company,
        "YearsInCurrentRole": years_in_current_role,
        "YearsSinceLastPromotion": years_since_last_promotion,
        "YearsWithCurrManager": years_with_curr_manager,
        "AttritionProbability_hidden": attrition_probability
    })

    return df


df = generate_ibm_hr_synthetic(n=100000, seed=42)
df.head()

In [0]:
# Removendo colunas com o mesmo valor para todos e identificadores
colunas_para_remover = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber', 'AttritionProbability_hidden']
df_clean = df.drop(columns=colunas_para_remover)

# Convertendo a variável Target (Attrition) para binário numérico (1 = Yes, 0 = No)
# Isso facilita muito a plotagem de gráficos e os modelos de Machine Learning futuros
df_clean['Attrition_Target'] = df_clean['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Dimensões do dataset após limpeza: {df_clean.shape[0]} linhas e {df_clean.shape[1]} colunas.")

In [0]:
# Calculando a taxa de Attrition
attrition_rate = df_clean['Attrition'].value_counts(normalize=True) * 100
print("Taxa de Rotatividade (Attrition):\n", attrition_rate.round(2))

# Gráfico de barras simples e elegante
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_clean, x='Attrition', palette=['#2ECC71', '#E74C3C'])
plt.title('Distribuição de Attrition (TechCorp Brasil)', fontsize=14, fontweight='bold')
plt.ylabel('Total de Funcionários')
plt.xlabel('Saiu da Empresa? (Attrition)')

# Adicionando os rótulos de dados no topo das barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Impacto da Renda Mensal
sns.boxplot(data=df_clean, x='Attrition', y='MonthlyIncome', palette='coolwarm', ax=axes[0])
axes[0].set_title('Renda Mensal vs Attrition')

# 2. Impacto da Idade
sns.kdeplot(data=df_clean, x='Age', hue='Attrition', fill=True, palette='coolwarm', ax=axes[1])
axes[1].set_title('Distribuição de Idade por Attrition')

# 3. Impacto de Horas Extras (OverTime)
# Calculando a porcentagem de attrition por quem faz hora extra
ot_attrition = df_clean.groupby('OverTime')['Attrition_Target'].mean() * 100
sns.barplot(x=ot_attrition.index, y=ot_attrition.values, palette='coolwarm', ax=axes[2])
axes[2].set_title('Taxa de Attrition vs Horas Extras')
axes[2].set_ylabel('% de Attrition')

plt.tight_layout()
plt.show()

In [0]:

# Configuração estética dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

# 1. Gráfico de Barras: Impacto de Hora Extra (OverTime) no Attrition
plt.figure(figsize=(8, 5))
sns.barplot(x='OverTime', y='Attrition_Target', data=df_clean, errorbar=None, palette='viridis')
plt.title('Taxa de Attrition por Realização de Hora Extra', fontsize=14, pad=15)
plt.xlabel('Faz Hora Extra?', fontsize=12)
plt.ylabel('Proporção de Attrition', fontsize=12)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
plt.show()

# 2. Boxplot: Distribuição de Renda Mensal por Attrition
plt.figure(figsize=(10, 6))
sns.boxplot(x='Attrition_Target', y='MonthlyIncome', data=df_clean, palette='Set2')
plt.title('Distribuição de Renda Mensal vs Attrition', fontsize=14, pad=15)
plt.xlabel('Attrition (0 = Ficou, 1 = Saiu)', fontsize=12)
plt.ylabel('Renda Mensal (R$)', fontsize=12)
plt.xticks([0, 1], ['Permaneceu', 'Desligado'])
plt.show()

In [0]:

print("--- VALIDAÇÃO ESTATÍSTICA DE INSIGHTS ---\n")

# A. Teste de Qui-Quadrado para OverTime (Categórica vs Categórica)
contingency_table = pd.crosstab(df_clean['OverTime'], df_clean['Attrition_Target'])
chi2, p_value_chi2, _, _ = stats.chi2_contingency(contingency_table)

print(f"1. Análise de Hora Extra (OverTime):")
print(f"   - Estatística Chi2: {chi2:.2f}")
print(f"   - P-Value: {p_value_chi2:.4e}")
if p_value_chi2 < 0.05:
    print("   -> Conclusão: Estatisticamente Significativo! Fazer hora extra afeta diretamente a decisão de sair.")
else:
    print("   -> Conclusão: Não há evidência estatística de impacto.")

print("\n" + "="*50 + "\n")

# B. Teste de Mann-Whitney U para MonthlyIncome (Contínua vs Categórica)
income_stay = df_clean[df_clean['Attrition_Target'] == 0]['MonthlyIncome']
income_leave = df_clean[df_clean['Attrition_Target'] == 1]['MonthlyIncome']
stat_mw, p_value_mw = stats.mannwhitneyu(income_stay, income_leave, alternative='two-sided')

print(f"2. Análise de Renda Mensal (MonthlyIncome):")
print(f"   - Estatística U: {stat_mw:.2f}")
print(f"   - P-Value: {p_value_mw:.4e}")
if p_value_mw < 0.05:
    print("   -> Conclusão: Estatisticamente Significativo! A distribuição salarial entre quem fica e quem sai é diferente.")
else:
    print("   -> Conclusão: Não há diferença salarial estatisticamente relevante.")

In [0]:
# Criando cópia segura para evitar alertas de cópia do pandas
df_featured = df_clean.copy()

# 1. Renda por ano de empresa (evitando divisão por zero)
df_featured['Income_Per_Year'] = df_featured['MonthlyIncome'] / (df_featured['YearsAtCompany'] + 1)

# 2. Índice de Estagnação (anos sem promoção em relação aos anos de casa)
df_featured['Stagnation_Index'] = df_featured['YearsSinceLastPromotion'] / (df_featured['YearsAtCompany'] + 1)

print("Novas features criadas com sucesso!")
print(df_featured[['MonthlyIncome', 'YearsAtCompany', 'Income_Per_Year', 'Stagnation_Index']].head())

In [0]:
# 1. Separando a variável alvo (y) das preditoras (X)
# Removemos o Attrition original (texto) e o Target (nossa resposta) da matriz X
X = df_featured.drop(columns=['Attrition_Target', 'Attrition']) 
y = df_featured['Attrition_Target']

# 2. Divisão Estratificada (Mantém os 12.5% de attrition tanto no treino quanto no teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Mapeando colunas automaticamente por tipo
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 4. Criando os transformadores do Pipeline
numerical_transformer = StandardScaler() # Ajusta a escala de salários, idades e dos novos índices
categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first') # Transforma textos em binário

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 5. Criando o Pipeline com o modelo de Regressão Logística
# Usamos class_weight='balanced' para o modelo dar mais peso à classe minoritária (quem sai)
model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

# 6. Treinando o modelo
model_lr.fit(X_train, y_train)

print("Pipeline executado e modelo de Regressão Logística treinado com sucesso!")
print(f"Registros de Treino: {X_train.shape[0]} | Registros de Teste: {X_test.shape[0]}")

In [0]:
# 1. Fazendo as predições de classe e de probabilidade
y_pred = model_lr.predict(X_test)
y_proba = model_lr.predict_proba(X_test)[:, 1]

# 2. Gerando a Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)

# 3. Calculando o AUC-ROC
auc = roc_auc_score(y_test, y_proba)

print("--- RELATÓRIO DE PERFORMANCE TÉCNICA ---\n")
print(classification_report(y_test, y_pred, target_names=['Permaneceu', 'Desligado']))
print(f"AUC-ROC Score: {auc:.4f}\n")

print("--- MATRIZ DE CONFUSÃO ---")
print(f"Verdadeiros Negativos (Ficaram e o modelo acertou): {cm[0][0]}")
print(f"Falsos Positivos (Ficaram, mas o modelo disse que iam sair): {cm[0][1]}")
print(f"Falsos Negativos (Saíram, mas o modelo não viu): {cm[1][0]}")
print(f"Verdadeiros Positivos (Saíram e o modelo acertou): {cm[1][1]}")

In [0]:
# Criando um DataFrame de simulação para o RH com a base de teste
df_rh = X_test.copy()
df_rh['Realidade'] = y_test
df_rh['Probabilidade_Risco'] = y_proba

# Ordenando pelos funcionários com maior risco de sair de acordo com o modelo
df_rh = df_rh.sort_values(by='Probabilidade_Risco', ascending=False)

# --- Premissas de Negócio dadas no Desafio ---
custo_total_attrition_empresa = 45000000.00  # R$ 45 milhões
meta_reducao = 0.10  # 10%
impacto_alvo = custo_total_attrition_empresa * meta_reducao  # R$ 4,5 milhões de economia desejada

# Vamos calcular o custo médio estimado por funcionário que sai na nossa base de teste
# Sabendo que a base de teste representa 20% do total da empresa
total_desligados_teste = y_test.sum()
custo_estimado_por_saida = (custo_total_attrition_empresa * 0.2) / total_desligados_teste

print("--- TRADUÇÃO FINANCEIRA PARA A DIRETORIA (BUSINESS CASE) ---\n")
print(f"Custo total atual da TechCorp com Attrition: R$ {custo_total_attrition_empresa:,.2f}")
print(f"Meta de Economia do Projeto (Redução de 10%): R$ {impacto_alvo:,.2f}")
print(f"Custo médio estimado por desligamento: R$ {custo_estimado_por_saida:,.2f}\n")

# Simulação de Ação Prática: O RH vai intervir nos TOP 10% funcionários com maior risco da base
top_criticos = int(len(df_rh) * 0.10)
df_top_criticos = df_rh.head(top_criticos)
assertivos_no_top = df_top_criticos['Realidade'].sum()

print(f"Se o RH focar uma ação preventiva nos top {top_criticos} colaboradores sob maior risco apontados pelo modelo:")
print(f"  - O modelo terá mapeado com precisão {assertivos_no_top} funcionários que realmente iriam sair.")

# Supondo que uma boa política de retenção (bônus, mudança de área, fim de hora extra) salve 60% dessas pessoas
taxa_sucesso_rh = 0.60
vidas_salvas = int(assertivos_no_top * taxa_sucesso_rh)
economia_gerada_teste = vidas_salvas * custo_estimado_por_saida
# Multiplicamos por 5 para expandir o resultado da base de teste (20%) para a empresa toda (100%)
economia_gerada_total = economia_gerada_teste * 5 

print(f"  - Se o RH conseguir reter {taxa_sucesso_rh*100}% deles, evitaremos {vidas_salvas * 5} demissões na TechCorp inteira.")
print(f"  - Economia total projetada para a empresa: R$ {economia_gerada_total:,.2f}")

if economia_gerada_total >= impacto_alvo:
    print(f"\n-> SUCESSO! O modelo supera a meta de R$ {impacto_alvo:,.2f} e gera um ROI positivo!")
else:
    print("\n-> O modelo ajuda, mas precisamos testar algoritmos mais complexos (como Random Forest) para alcançar a meta financeira.")

In [0]:
# 1. Criar o Pipeline com o mesmo pré-processador, mudando apenas o classificador
# Usamos class_weight='balanced' para tratar os mesmos 12.5% de desbalanceamento
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        class_weight='balanced', 
        random_state=42, 
        n_jobs=-1
    ))
])

# 2. Treinar o modelo de Random Forest
model_rf.fit(X_train, y_train)

# 3. Fazer as predições de classe e probabilidade para a base de teste
y_pred_rf = model_rf.predict(X_test)
y_proba_rf = model_rf.predict_proba(X_test)[:, 1]

# 4. Gerar a Matriz de Confusão e AUC-ROC
cm_rf = confusion_matrix(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print("--- PERFORMANCE TÉCNICA: RANDOM FOREST ---\n")
print(classification_report(y_test, y_pred_rf, target_names=['Permaneceu', 'Desligado']))
print(f"AUC-ROC Score (Random Forest): {auc_rf:.4f}\n")

print("--- MATRIZ DE CONFUSÃO (RANDOM FOREST) ---")
print(f"Verdadeiros Negativos (Ficaram e o modelo acertou): {cm_rf[0][0]}")
print(f"Falsos Positivos (Ficaram, mas o modelo disse que iam sair): {cm_rf[0][1]}")
print(f"Falsos Negativos (Saíram, mas o modelo não viu): {cm_rf[1][0]}")
print(f"Verdadeiros Positivos (Saíram e o modelo acertou): {cm_rf[1][1]}")

In [0]:
# Criando o DataFrame de simulação para a Random Forest
df_rh_rf = X_test.copy()
df_rh_rf['Realidade'] = y_test
df_rh_rf['Probabilidade_Risco'] = y_proba_rf

# Ordenando pelos funcionários com maior risco de acordo com a Random Forest
df_rh_rf = df_rh_rf.sort_values(by='Probabilidade_Risco', ascending=False)

# Usando as mesmas premissas anteriores
custo_total_attrition_empresa = 45000000.00
custo_estimado_por_saida = (custo_total_attrition_empresa * 0.2) / y_test.sum()

# Intervenção nos TOP 10% de maior risco da Random Forest
top_criticos_rf = int(len(df_rh_rf) * 0.10)
df_top_criticos_rf = df_rh_rf.head(top_criticos_rf)
assertivos_no_top_rf = df_top_criticos_rf['Realidade'].sum()

print("--- TRADUÇÃO FINANCEIRA: RANDOM FOREST ---\n")
print(f"Ao focar nos top {top_criticos_rf} colaboradores sob maior risco da Random Forest:")
print(f"  - O modelo mapeou com precisão {assertivos_no_top_rf} funcionários que realmente iriam sair.")

# Mantendo a taxa de sucesso de 60% do RH
taxa_sucesso_rh = 0.60
vidas_salvas_rf = int(assertivos_no_top_rf * taxa_sucesso_rh)
economia_rf_total = vidas_salvas_rf * custo_estimado_por_saida * 5

print(f"  - Evitaremos {vidas_salvas_rf * 5} demissões na TechCorp inteira.")
print(f"  - Economia total projetada com Random Forest: R$ {economia_rf_total:,.2f}")

In [0]:
print("--- REFINAMENTO DA EDA & TRATAMENTO DE OUTLIERS ---\n")

# Copiando a base para aplicar as correções
df_refined = df_featured.copy()

# 1. Investigando Outliers via IQR para as variáveis numéricas chave
cols_para_analise = ['MonthlyIncome', 'YearsAtCompany', 'Age', 'Income_Per_Year']

for col in cols_para_analise:
    Q1 = df_refined[col].quantile(0.25)
    Q3 = df_refined[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers_count = df_refined[(df_refined[col] < limite_inferior) | (df_refined[col] > limite_superior)].shape[0]
    print(f"Variável '{col}':")
    print(f"  - Limite Inferior: {limite_inferior:.2f} | Limite Superior: {limite_superior:.2f}")
    print(f"  - Quantidade de Outliers detectados: {outliers_count} ({outliers_count/len(df_refined)*100:.2f}%)")

# 2. Estratégia de Tratamento: Transformação Logarítmica para Renda (suaviza a cauda longa de salários altos)
# Adicionamos +1 para evitar log(0) caso existam valores zerados
df_refined['Log_MonthlyIncome'] = np.log1p(df_refined['MonthlyIncome'])
df_refined['Log_Income_Per_Year'] = np.log1p(df_refined['Income_Per_Year'])

# 3. Tratamento por Capping (Winsorization) para YearsAtCompany
# Quem tiver tempo de empresa acima do limite superior, trazemos para o valor do limite (evita distorcer o modelo linear)
Q1_years = df_refined['YearsAtCompany'].quantile(0.25)
Q3_years = df_refined['YearsAtCompany'].quantile(0.75)
IQR_years = Q3_years - Q1_years
limite_sup_years = Q3_years + 1.5 * IQR_years

df_refined['YearsAtCompany_Capped'] = np.where(df_refined['YearsAtCompany'] > limite_sup_years, limite_sup_years, df_refined['YearsAtCompany'])

# Removendo as colunas antigas que foram transformadas para evitar redundância
df_refined = df_refined.drop(columns=['MonthlyIncome', 'YearsAtCompany', 'Income_Per_Year'])

print("\nTratamento de outliers concluído com sucesso!")
print(f"Novas colunas criadas: ['Log_MonthlyIncome', 'Log_Income_Per_Year', 'YearsAtCompany_Capped']")

In [0]:
# 1. Definindo as variáveis preditoras (X) e a alvo (y) usando a base refinada
X = df_refined.drop(columns=['Attrition_Target', 'Attrition']) 
y = df_refined['Attrition_Target']

# 2. Divisão Estratificada (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Mapeando as colunas automaticamente (as colunas antigas já foram removidas no passo anterior)
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 4. Criando os transformadores do Pipeline
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 5. Configurando o LightGBM com pesos balanceados cirurgicamente
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

model_lgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        n_estimators=200,       # Subimos para 200 árvores para capturar mais padrões
        learning_rate=0.03,     # Passo menor para um aprendizado mais preciso
        max_depth=6,            # Evita árvores profundas demais (overfitting)
        num_leaves=31,          # Limite de nós por árvore
        scale_pos_weight=ratio, # Balanceamento exato para os 12.5% de attrition
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Treinando o modelo robusto
model_lgb.fit(X_train, y_train)

# 7. Fazendo as predições de classe e probabilidade
y_pred_lgb = model_lgb.predict(X_test)
y_proba_lgb = model_lgb.predict_proba(X_test)[:, 1]

# 8. Avaliação das Métricas Técnicas
auc_lgb = roc_auc_score(y_test, y_proba_lgb)
cm_lgb = confusion_matrix(y_test, y_pred_lgb)

print("--- PERFORMANCE DO MODELO AVANÇADO (LIGHTGBM + DADOS REFINADOS) ---\n")
print(classification_report(y_test, y_pred_lgb, target_names=['Permaneceu', 'Desligado']))
print(f"NOVO AUC-ROC Score: {auc_lgb:.4f}\n")

print("--- MATRIZ DE CONFUSÃO (LIGHTGBM) ---")
print(f"Verdadeiros Negativos (Ficaram e acertou): {cm_lgb[0][0]}")
print(f"Falsos Positivos (Alarme falso): {cm_lgb[0][1]}")
print(f"Falsos Negativos (Deixou passar): {cm_lgb[1][0]}")
print(f"Verdadeiros Positivos (Saíram e acertou): {cm_lgb[1][1]}")

In [0]:
# 1. Calculando a matriz de correlação apenas para as colunas numéricas refinadas
num_cols_refined = df_refined.select_dtypes(include=['int64', 'float64']).columns
corr_matrix = df_refined[num_cols_refined].corr()

# 2. Isolando a correlação das variáveis especificamente com o Attrition Target
# Ordenamos do maior impacto positivo para o maior impacto negativo
attrition_corr = corr_matrix[['Attrition_Target']].sort_values(by='Attrition_Target', ascending=False)

# 3. Removendo a própria linha do Attrition_Target para não dar correlação 1.0 óbvia
attrition_corr = attrition_corr.drop('Attrition_Target')

# 4. Plotando um Heatmap focado e vertical (Muito mais fácil de ler no relatório!)
plt.figure(figsize=(6, 10))
sns.heatmap(
    attrition_corr, 
    annot=True, 
    fmt=".2f",          # Limita a 2 casas decimais para não encavalar o texto!
    cmap='coolwarm', 
    vmin=-0.5, vmax=0.5, # Ajusta os limites para destacar forças moderadas
    linewidths=0.5,
    cbar_kws={'label': 'Coeficiente de Correlação'}
)

plt.title('Correlação das Variáveis Numéricas\ncom o Attrition (Alvo)', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

In [0]:
# 1. Copiando a base que já veio com os outliers tratados
df_extreme = df_refined.copy()

# 2. Eliminando colunas lixo/constantes se elas ainda existirem
colunas_para_remover = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_extreme = df_extreme.drop(columns=[col for col in colunas_para_remover if col in df_extreme.columns])

# 3. ENGENHARIA DE RECURSOS AVANÇADA (Cruzando Categóricas e Numéricas)

# Feature 1: Alerta de Burnout (Hora extra + Baixa Satisfação com o Clima)
df_extreme['Burnout_Alert'] = np.where(
    (df_extreme['OverTime'] == 'Yes') & (df_extreme['EnvironmentSatisfaction'] <= 2), 1, 0
)

# Feature 2: Desgaste por Viagem (Viaja Muito + WorkLifeBalance Ruim)
df_extreme['Travel_Fatigue'] = np.where(
    (df_extreme['BusinessTravel'] == 'Travel_Frequently') & (df_extreme['WorkLifeBalance'] <= 2), 1, 0
)

# Feature 3: Casado ou Solteiro com Estabilidade Financeira
df_extreme['Single_Low_Income'] = np.where(
    (df_extreme['MaritalStatus'] == 'Single') & (df_extreme['Log_MonthlyIncome'] < df_extreme['Log_MonthlyIncome'].median()), 1, 0
)

print(f"Novas features estratégicas criadas! Total de colunas atual: {df_extreme.shape[1]}")

# 4. Novo Split dos Dados
X_ex = df_extreme.drop(columns=['Attrition_Target', 'Attrition'], errors='ignore')
y_ex = df_extreme['Attrition_Target']

X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(
    X_ex, y_ex, test_size=0.2, random_state=42, stratify=y_ex
)

# 5. Pipeline de Processamento Automatizado
num_cols_ex = X_ex.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols_ex = X_ex.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor_ex = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_ex),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols_ex)
    ]
)

# 6. LightGBM Tunado para Maximização de AUC
ratio_ex = (len(y_train_ex) - sum(y_train_ex)) / sum(y_train_ex)

model_lgb_extreme = Pipeline(steps=[
    ('preprocessor', preprocessor_ex),
    ('classifier', LGBMClassifier(
        n_estimators=300,         # Mais árvores para aprender os cruzamentos complexos
        learning_rate=0.02,       # Aprendizado mais lento e cirúrgico
        max_depth=7,              # Um pouco mais de profundidade para capturar as interações
        num_leaves=50,            # Mais nós para acomodar as novas regras
        subsample=0.8,            # Amostragem de linhas para evitar memorização (overfit)
        colsample_bytree=0.8,     # Amostragem de colunas por árvore
        scale_pos_weight=ratio_ex,
        random_state=42,
        n_jobs=-1
    ))
])

# 7. Treinar o novo modelo
model_lgb_extreme.fit(X_train_ex, y_train_ex)

# 8. Avaliação
y_pred_ex = model_lgb_extreme.predict(X_test_ex)
y_proba_ex = model_lgb_extreme.predict_proba(X_test_ex)[:, 1]

auc_ex = roc_auc_score(y_test_ex, y_proba_ex)

print("\n--- PERFORMANCE DO MODELO ULTRA REFINADO ---")
print(f"NOVO AUC-ROC SCORE: {auc_ex:.4f}\n")
print(classification_report(y_test_ex, y_pred_ex, target_names=['Permaneceu', 'Desligado']))

In [0]:

# 1. CONFIGURAÇÃO DO ESTILO VISUAL: Adotando um tema 'clean' e profissional (estilo whitegrid)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(11, 6.5))

# 2. DEFINIÇÃO DOS EIXOS: Criação das labels dos 10 decis de risco (de 10% em 10%)
decis_percentuais = [f'{i*10}% a {(i+1)*10}%' for i in range(10)]

# 3. PLOTAGEM DA CURVA PRINCIPAL: Linha verde corporativa (#1b5e20) representando a economia do modelo
ax.plot(decis_percentuais, economia_acumulada, marker='o', markersize=8, 
        linewidth=3.5, color='#1b5e20', label='Economia Acumulada Projetada')

# 4. PLOTAGEM DA META: Linha tracejada vermelha (#d32f2f) indicando o objetivo estipulado pela diretoria
ax.axhline(y=impacto_alvo, color='#d32f2f', linestyle='--', linewidth=2.5, 
           label='Meta de Economia da Diretoria (R$ 4,5M)')

# 5. CUSTOMIZAÇÃO DOS TÍTULOS E LABELS: Textos claros com fontes visíveis para apresentação executiva
ax.set_title('Curva de Ganho Financeiro Acumulado (ROI)\nImpacto de Negócio por Decil de Risco do Modelo', 
             fontsize=15, pad=20, fontweight='bold', color='#333333')
ax.set_xlabel('Percentual de Funcionários Abordados (Ordenados do Maior Risco para o Menor)', 
              fontsize=12, labelpad=12, color='#333333')
ax.set_ylabel('Economia Total Projetada (Em Reais)', fontsize=12, labelpad=12, color='#333333')

# 6. FORMATAÇÃO DO EIXO Y: Transforma os números puros em formato de moeda brasileira (R$)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.tick_params(axis='both', labelsize=11)

# 7. SUAVIZAÇÃO DA GRADE: Linhas de fundo pontilhadas e discretas para não poluir o gráfico
ax.grid(True, linestyle='--', alpha=0.5, color='#cccccc')

# 8. ANOTAÇÃO DO DESTAQUE (TOP 10%): Criação de um balão de destaque (bbox) elegante e bem posicionado
ax.annotate(
    f'Top 10% Mais Críticos\nEconomia: R$ {economia_acumulada[0]:,.2f}\n(Quase o dobro da meta!)', 
    xy=(0, economia_acumulada[0]), 
    xytext=(0.8, economia_acumulada[0] + 2500000), # Desloca o texto para cima e para a direita dando respiro visual
    arrowprops=dict(arrowstyle="->", color='#333333', lw=1.5, connectionstyle="arc3,rad=-0.2"), # Seta sutil e curvada
    fontsize=11, fontweight='bold', color='#1b5e20',
    bbox=dict(boxstyle="round,pad=0.5", fc="#e8f5e9", ec="#1b5e20", lw=1) # Caixa verde clara de destaque
)

# 9. POSICIONAMENTO DA LEGENDA: Alocada no canto superior esquerdo para não cobrir a linha da meta
ax.legend(fontsize=11, loc='upper left', frameon=True, facecolor='white', edgecolor='#cccccc')

# 10. POLIMENTO ESTÉTICO (DESPINE): Remove as bordas desnecessárias do gráfico (padrão de design clean)
if 'sns' in globals() or 'seaborn' in globals():
    sns.despine(left=True, bottom=True)

# 11. EXIBIÇÃO: Ajusta o espaçamento dos elementos antes de plotar na tela
plt.tight_layout()
plt.show()